# Fleet Maintenance Planning — LightGBM Version 1

This notebook builds one Kaggle-ready classical machine-learning pipeline using LightGBM.

**Competition objective:** minimise total maintenance cost, not classification error.

- `flag = 1`: inspect the vehicle.
- `flag = 0`: do not inspect.
- `budget`: spare-parts value to pre-order; ignored when `flag = 0`.

The notebook creates `submission_v1_lightgbm.csv` and evaluates the inspection policy using validation maintenance cost.


## 1. Imports and reproducibility

This notebook uses LightGBM for failure classification and conditional repair-cost estimation.

In [ ]:
import os, zipfile, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

from lightgbm import LGBMClassifier, LGBMRegressor, early_stopping, log_evaluation

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")

## 2. Load the competition data

Put `train.csv`, `test.csv`, and `sample_submission.csv` in the notebook working directory. Alternatively, put the uploaded ZIP file there; the cell below extracts it.

In [ ]:
ZIP_PATH = "mlprojectfleet2.zip"

if os.path.exists(ZIP_PATH) and not os.path.exists("train.csv"):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

sensor_cols = [c for c in train.columns if c.startswith("s")]
# s047 is entirely missing in the supplied test set, so it cannot provide
# useful test-time information. Drop it consistently from train and test.
feature_cols = [c for c in sensor_cols if c in test.columns]
feature_cols = [c for c in feature_cols if not test[c].isna().all()]

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train["failure"].astype(int)
repair_cost = train["repair_cost"].fillna(0).astype(float)

print("Train:", train.shape, " Test:", test.shape)
print("Sensor features used:", len(feature_cols))
print("Failure count:", int(y.sum()), "/", len(y))
print("Failure rate:", round(y.mean() * 100, 3), "%")
print("Positive repair-cost rows:", int((train.failure == 1).sum()))
print("Test IDs unique:", test.id.is_unique)

## 3. Initial data inspection

The training file has 56,000 rows and 170 sensor columns. Failure is rare, so accuracy alone is misleading. `followup_days` is not included in the feature matrix because it is absent from test data.

Check missingness and the repair-cost distribution before modelling.

In [ ]:
print(train[["failure", "followup_days", "repair_cost"]].describe(include="all"))
print("\nFailure counts:\n", y.value_counts())
print("\nLargest sensor missingness rates:\n",
      train[feature_cols].isna().mean().sort_values(ascending=False).head(15))

plt.figure(figsize=(6, 4))
y.value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["No failure", "Failure"], rotation=0)
plt.ylabel("Vehicles")
plt.title("Failure class distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
train.loc[train.failure == 1, "repair_cost"].clip(upper=3000).hist(bins=40)
plt.xlabel("Repair cost (clipped at 3000 for display)")
plt.ylabel("Failed vehicles")
plt.title("Repair-cost distribution")
plt.tight_layout()
plt.show()

## 4. Validation split and the competition cost function

We use a stratified holdout split so the rare failure class appears in both sets. This is a useful first validation estimate, but it cannot fully reproduce the shift to a different depot.

The cost function below exactly follows the project statement:

- no inspection: actual repair cost;
- inspection: 10 + 0.20 × under-budget amount + 0.02 × over-budget amount.

For healthy vehicles, actual repair cost is zero.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
c_valid = train.loc[y_valid.index, "repair_cost"].fillna(0).to_numpy(dtype=float)

def competition_cost(y_true_cost, flags, budgets):
    y_true_cost = np.asarray(y_true_cost, dtype=float)
    flags = np.asarray(flags, dtype=int)
    budgets = np.asarray(budgets, dtype=float)
    per_vehicle = np.where(
        flags == 1,
        10 + 0.20 * np.maximum(y_true_cost - budgets, 0)
          + 0.02 * np.maximum(budgets - y_true_cost, 0),
        y_true_cost
    )
    return per_vehicle.mean(), per_vehicle.sum()

def choose_policy(expected_repair_cost, budgets_for_flagged, actual_cost, label="model"):
    # Search decision thresholds using validation only.
    # The final chosen threshold is then applied unchanged to test predictions.
    thresholds = np.unique(np.quantile(expected_repair_cost, np.linspace(0.70, 0.999, 100)))
    rows = []
    for threshold in thresholds:
        flags = (expected_repair_cost > threshold).astype(int)
        budgets = np.where(flags == 1, budgets_for_flagged, 0.0)
        mean_cost, total_cost = competition_cost(actual_cost, flags, budgets)
        rows.append((threshold, int(flags.sum()), mean_cost, total_cost))
    results = pd.DataFrame(rows, columns=["threshold", "inspections", "mean_cost", "total_cost"])
    best = results.loc[results.mean_cost.idxmin()]
    print(f"{label}: best threshold={best.threshold:.4f}, "
          f"inspections={int(best.inspections)}, mean cost={best.mean_cost:.4f}")
    return float(best.threshold), results

# Simple budget baseline: 90th percentile of observed repair costs among failed
# training vehicles. Later versions can improve this with conditional quantiles.
global_q90 = float(train.loc[train.failure == 1, "repair_cost"].quantile(0.90))
print("Training failure rate:", y_train.mean())
print("Global 90th-percentile repair budget:", round(global_q90, 2))

## 5. Model Version 1 — LightGBM

LightGBM can model nonlinear sensor interactions and handles missing values natively. We use a positive-class weight to help the model learn from the rare failures.

The repair-cost model is trained only on failed vehicles, using log-transformed repair costs to reduce the influence of the long right tail.

In [ ]:
scale_pos = max(1.0, (y_train == 0).sum() / max(1, (y_train == 1).sum()))

v1_failure = LGBMClassifier(
    n_estimators=1200, learning_rate=0.025, num_leaves=15,
    max_depth=-1, min_child_samples=35, subsample=0.85,
    colsample_bytree=0.85, reg_lambda=5.0,
    scale_pos_weight=scale_pos, random_state=SEED,
    verbosity=-1
)
v1_failure.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[early_stopping(100, verbose=False), log_evaluation(0)]
)
p1 = v1_failure.predict_proba(X_valid)[:, 1]

v1_cost = LGBMRegressor(
    n_estimators=700, learning_rate=0.025, num_leaves=7,
    max_depth=3, min_child_samples=12, subsample=0.9,
    colsample_bytree=0.85, reg_lambda=8.0,
    random_state=SEED, verbosity=-1
)
v1_cost.fit(
    X_train_pos, log_cost_train,
    eval_set=[(X_train_pos, log_cost_train)],
    callbacks=[log_evaluation(0)]
)
r1 = np.maximum(np.expm1(v1_cost.predict(X_valid)), 0)
expected1 = p1 * r1

threshold1, results1 = choose_policy(
    expected1, global_q90, c_valid, "Version 1"
)
flags1 = (expected1 > threshold1).astype(int)
budget1 = np.where(flags1 == 1, global_q90, 0.0)
v1_mean_cost, v1_total_cost = competition_cost(c_valid, flags1, budget1)

print("V1 average precision:", round(average_precision_score(y_valid, p1), 5))
print("V1 ROC-AUC:", round(roc_auc_score(y_valid, p1), 5))
print("V1 validation mean cost:", round(v1_mean_cost, 5))

## 6. Refit Version 1 on all training data and create the submission

Fit both LightGBM models on all available training rows, apply the validation-selected inspection threshold, and save the Version 1 submission.


In [ ]:
# Full-data feature matrices
X_full = X
X_test_full = X_test
y_full = y
positive_full = (y_full == 1)
X_positive_full = X_full.loc[positive_full]
log_cost_full = np.log1p(train.loc[X_positive_full.index, "repair_cost"].astype(float))

v1_failure_full = LGBMClassifier(
    n_estimators=max(200, getattr(v1_failure, "best_iteration_", 0) or 700),
    learning_rate=0.025, num_leaves=15, min_child_samples=35,
    subsample=0.85, colsample_bytree=0.85, reg_lambda=5.0,
    scale_pos_weight=max(1.0, (y_full == 0).sum() / max(1, (y_full == 1).sum())),
    random_state=SEED, verbosity=-1
)
v1_failure_full.fit(X_full, y_full)
v1_cost_full = LGBMRegressor(
    n_estimators=700, learning_rate=0.025, num_leaves=7, max_depth=3,
    min_child_samples=12, subsample=0.9, colsample_bytree=0.85,
    reg_lambda=8.0, random_state=SEED, verbosity=-1
)
v1_cost_full.fit(X_positive_full, log_cost_full)
p1_test = v1_failure_full.predict_proba(X_test_full)[:, 1]
r1_test = np.maximum(np.expm1(v1_cost_full.predict(X_test_full)), 0)
e1_test = p1_test * r1_test

def make_submission(expected_cost, threshold, filename):
    flags = (expected_cost > threshold).astype(int)
    budgets = np.where(flags == 1, global_q90, 0.0)
    sub = pd.DataFrame({"id": test["id"], "flag": flags, "budget": budgets})
    sub["flag"] = sub["flag"].astype(int)
    sub.to_csv(filename, index=False)
    assert len(sub) == len(test)
    assert sub["id"].is_unique
    assert set(sub["flag"].unique()).issubset({0, 1})
    assert np.isfinite(sub["budget"]).all() and (sub["budget"] >= 0).all()
    print(filename, "| inspections:", int(flags.sum()), "| mean budget among flagged:",
          round(sub.loc[sub.flag == 1, "budget"].mean(), 2) if flags.sum() else 0)
    return sub

sub1 = make_submission(e1_test, threshold1, "submission_v1_lightgbm.csv")


## 7. Final checks and files

Before submitting, verify the row count, columns, IDs, flags, and budgets.


In [ ]:
name = "submission_v1_lightgbm.csv"
check = pd.read_csv(name)
print(name, check.shape, list(check.columns),
      "valid flags:", check.flag.isin([0, 1]).all(),
      "valid budgets:", np.isfinite(check.budget).all() and (check.budget >= 0).all())
